# Notebook 01 — Data & Feature Collecting (`store_id = 5631`)

**Mục tiêu (Luồng ML Thống nhất `01 → 05`):**
1. Kết nối tới tầng Gold (`quick_gold_md01`) lấy 2 bảng Mart: `mart_sales_observation_period` (`OBSERVED` 8 nhóm ATC) và `mart_sales_weather_daily` (Open-Meteo ERA5).
2. Thu thập nguồn ngoại sinh **Lịch nghỉ lễ chính thức Serbia (`holidays.country_holidays('RS')`)** giai đoạn `2014–2019`.
3. Thực thi **Data Contract Audit**: kiểm tra `provenance_class == 'OBSERVED'`, 0 trùng khóa `(atc_code, local_date)`, 0 ngày thiếu trên lưới liên tục `2014-01-02 → 2019-10-08`, `qty >= 0`, và mốc bắt đầu thời tiết (`2014-01-02`) để bảo đảm chỉ cần `.ffill()`.
4. Lưu snapshot bất biến (`raw_*.parquet`) + `data_manifest.json` (kèm SHA-256) cho Notebook 02–05 chạy offline.

In [ ]:
# Nếu chạy lần đầu trên Google Colab Cloud, bỏ comment dòng dưới để cài thư viện
# !pip install -q "psycopg[binary]" python-dotenv pandas numpy holidays pyarrow

import hashlib
import json
import warnings
from datetime import datetime, timezone

import holidays
import pandas as pd
import psycopg

warnings.filterwarnings("ignore")

from pathlib import Path
import os

def resolve_art_dir() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "BTL_BI").is_dir():
            d = p / "BTL_BI" / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
        if p.name == "BTL_BI":
            d = p / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = cwd / "artifacts" / "forecasting"
    d.mkdir(parents=True, exist_ok=True)
    return d

ART_DIR = resolve_art_dir()
print("Artifacts dir:", ART_DIR)

def load_neon_dsn() -> str:
    try:
        from dotenv import load_dotenv
        cwd = Path.cwd().resolve()
        for p in [cwd, *cwd.parents]:
            env_file = p / ".env"
            if env_file.is_file():
                load_dotenv(env_file)
                break
    except Exception:
        pass
    dsn = os.environ.get("QUICK_NEON_DATABASE_URL") or ""
    if not dsn:
        try:
            from google.colab import userdata
            dsn = userdata.get("QUICK_NEON_DATABASE_URL") or ""
        except Exception:
            pass
    if not dsn:
        import getpass
        dsn = getpass.getpass("Nhập QUICK_NEON_DATABASE_URL: ")
    assert dsn, "Missing QUICK_NEON_DATABASE_URL"
    return dsn

In [ ]:
SQL_SALES = '''
SELECT period_start::date AS local_date,
       category_code AS atc_code,
       category_name,
       provenance_class,
       source,
       subject_kind,
       observed_qty::double precision AS qty
FROM quick_gold_md01.mart_sales_observation_period
WHERE source = 'PHARMA_SALES_DAILY_ATC'
  AND store_id = 5631
  AND subject_kind = 'CATEGORY'
ORDER BY period_start, category_code;
'''

SQL_WEATHER = '''
SELECT DISTINCT ON (local_date)
       local_date::date AS local_date,
       temp_mean, temp_min, temp_max, apparent_temp_mean,
       humidity_mean, precip_sum, wind_speed_max, wind_speed_mean,
       cloud_cover_mean, pressure_msl_mean, uv_index_max,
       shortwave_radiation_sum, daylight_duration_s
FROM quick_gold_md01.mart_sales_weather_daily
WHERE store_id = 5631
  AND has_weather_observation = TRUE
ORDER BY local_date, weather_retrieved_at_utc DESC NULLS LAST;
'''

def fetch_df(conn, sql: str) -> pd.DataFrame:
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

dsn = load_neon_dsn()
with psycopg.connect(dsn) as conn:
    df_sales = fetch_df(conn, SQL_SALES)
    df_weather = fetch_df(conn, SQL_WEATHER)
del dsn

df_sales["local_date"] = pd.to_datetime(df_sales["local_date"])
df_weather["local_date"] = pd.to_datetime(df_weather["local_date"])

rs_hol = holidays.country_holidays("RS", years=range(2014, 2020))
df_hol = pd.DataFrame(
    [{"local_date": pd.Timestamp(d), "holiday_name": name} for d, name in sorted(rs_hol.items())]
)

print(f"Sales rows: {len(df_sales):,}, Weather rows: {len(df_weather):,}, Holidays rows: {len(df_hol):,}")

In [ ]:
expected_days = pd.date_range("2014-01-02", "2019-10-08", freq="D")
assert len(expected_days) == 2106
assert len(df_sales) == 16848
assert set(df_sales["provenance_class"].unique()) == {"OBSERVED"}
assert set(df_sales["source"].unique()) == {"PHARMA_SALES_DAILY_ATC"}
assert set(df_sales["subject_kind"].unique()) == {"CATEGORY"}
assert not df_sales.duplicated(["atc_code", "local_date"]).any()
assert df_sales["qty"].isna().sum() == 0 and (df_sales["qty"] >= 0).all()

assert not df_weather.duplicated(["local_date"]).any()
assert df_weather["local_date"].min() == expected_days[0]
assert df_weather["local_date"].max() == expected_days[-1]

audit_rows = []
for atc, g in df_sales.groupby("atc_code"):
    missing_cnt = len(set(expected_days) - set(g["local_date"]))
    assert missing_cnt == 0
    audit_rows.append({
        "atc_code": atc,
        "rows": len(g),
        "min_date": str(g["local_date"].min().date()),
        "max_date": str(g["local_date"].max().date()),
        "missing_days": missing_cnt,
        "dup_keys": int(g.duplicated(["local_date"]).sum()),
        "zero_days": int((g["qty"] == 0).sum()),
        "qty_sum": round(float(g["qty"].sum()), 2),
    })

df_audit = pd.DataFrame(audit_rows)
print("=== DATA CONTRACT AUDIT: PASS ===")
display(df_audit)

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

p_sales = ART_DIR / "raw_sales_5631.parquet"
p_weather = ART_DIR / "raw_weather_5631.parquet"
p_hol = ART_DIR / "raw_holidays_rs.parquet"

df_sales.to_parquet(p_sales, index=False)
df_weather.to_parquet(p_weather, index=False)
df_hol.to_parquet(p_hol, index=False)

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "store_id": 5631,
    "provenance_class": "OBSERVED",
    "source": "PHARMA_SALES_DAILY_ATC",
    "date_range": ["2014-01-02", "2019-10-08"],
    "expected_calendar_days": int(len(expected_days)),
    "weather_observed_days": int(len(df_weather)),
    "missing_weather_days": int(len(expected_days) - len(df_weather)),
    "files": {
        p.name: {"rows": int(cnt), "sha256": sha256_file(p)}
        for p, cnt in [(p_sales, len(df_sales)), (p_weather, len(df_weather)), (p_hol, len(df_hol))]
    },
}
(ART_DIR / "data_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved immutable snapshots + data_manifest.json successfully.")